---
title: "广度优先搜索 (BFS)——层级遍历"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [2]:
#| code-fold: true
from collections import deque
from typing import List, Optional
class TreeNode:
    def __init__(self, val: int = 0, left: 'TreeNode' = None, right: 'TreeNode' = None):
        self.val = val
        self.left = left
        self.right = right

## 📝 题目 102：二叉树的层序遍历

::: {.callout-caution}
### 题目要求
**描述**：给你二叉树的根节点 `root` ，返回其节点值的 **层序遍历** 。即逐层地，从左到右访问所有节点）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`root = [3,9,20,null,null,15,7]`
    * **输出**：`[[3],[9,20],[15,7]]`

* **示例 2**：
    * **输入**：`root = [1]`
    * **输出**：`[[1]]`

* **示例 3**：
    * **输入**：`root = []`
    * **输出**：`[]`
:::

---

In [3]:
class Solution102:
    def levelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if not root:  # 边界处理：如果根节点为空，直接返回空列表
            return []
        queue = deque([root])
        result = []
        while queue:
            level = []
            for _ in range(len(queue)):  # 锁定当前层的节点数量（这是层序遍历的核心）
                node = queue.popleft()
                level.append(node.val)
                if node.left:  # 将下一层的子节点（如果有）加入队列末尾
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            result.append(level)
        return result

In [4]:
#| code-fold: true
def build_tree(data):
    """将 LeetCode 格式的列表转换为二叉树对象"""
    if not data: return None
    root = TreeNode(data[0])
    queue = deque([root])
    i = 1
    while queue and i < len(data):
        curr = queue.popleft()
        if i < len(data) and data[i] is not None:
            curr.left = TreeNode(data[i])
            queue.append(curr.left)
        i += 1
        if i < len(data) and data[i] is not None:
            curr.right = TreeNode(data[i])
            queue.append(curr.right)
        i += 1
    return root

def test_102(func):
    test_cases = [
        {"input": [3,9,20,None,None,15,7], "expected": [[3],[9,20],[15,7]], "desc": "标准平衡树"},
        {"input": [1,2,None,3,None,4,None,5], "expected": [[1],[2],[3],[4],[5]], "desc": "极度倾斜树（全左）"},
        {"input": [1], "expected": [[1]], "desc": "单节点树"},
        {"input": [], "expected": [], "desc": "空树测试"},
        {"input": [1,None,2,None,3], "expected": [[1],[2],[3]], "desc": "右倾斜树"},
        {"input": [1,2,3,4,None,None,5], "expected": [[1],[2,3],[4,5]], "desc": "带空缺的普通树"},
        {"input": [1,2,2,3,4,4,3], "expected": [[1],[2,2],[3,4,4,3]], "desc": "对称二叉树"},
        {"input": [0,None,None], "expected": [[0]], "desc": "值为0的节点测试"},
        {"input": [1,2,3,4,5,6,7], "expected": [[1],[2,3],[4,5,6,7]], "desc": "满二叉树"},
        {"input": [1,None,2,3,None,None,4], "expected": [[1],[2],[3],[4]], "desc": "复杂空缺路径"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 50)

    for i, tc in enumerate(test_cases):
        root = build_tree(tc["input"])
        actual = func(root)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
        if not is_pass:
            print(f"   ❌ 预期: {tc['expected']}, 实际: {actual}")
    print("-" * 50)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_102(Solution102().levelOrder)

ID   | 结果     | 描述
--------------------------------------------------
1    | ✅ PASS | 标准平衡树
2    | ✅ PASS | 极度倾斜树（全左）
3    | ✅ PASS | 单节点树
4    | ✅ PASS | 空树测试
5    | ✅ PASS | 右倾斜树
6    | ✅ PASS | 带空缺的普通树
7    | ✅ PASS | 对称二叉树
8    | ✅ PASS | 值为0的节点测试
9    | ✅ PASS | 满二叉树
10   | ✅ PASS | 复杂空缺路径
--------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **初始化与特殊情况判定**：
   - 检查根节点 `root` 是否为空。若为空，则二叉树没有任何节点，直接返回空列表 `[]`。
   - 创建队列 `q`，将根节点 `root` 入队。这是搜索的起点。

2. **层级处理逻辑（关键）**：
   - 创建结果列表 `res`，用于存储每一层的结果。
   - **循环条件**：只要队列不为空，说明还有节点待访问。
   - **锁定当前层大小**：在开始处理每一层之前，先记录当前队列的长度 `size = len(q)`。这个 `size` 代表了**当前这一层**所拥有的节点个数。
   - **分层处理**：执行 `for _ in range(size)` 循环。
     - 弹出队首节点 `node`。
     - 将 `node.val` 加入到当前层的临时列表 `level` 中。
     - **扩展下一层**：
       - 如果该节点有左孩子，将左孩子入队。
       - 如果该节点有右孩子，将右孩子入队。
     - *注意*：这些新加入的子节点排在队列末尾，由于我们锁定了 `size`，它们不会在当前轮 `for` 循环中被处理，而是留到下一轮 `while` 循环。

3. **收集结果**：
   - 当 `for` 循环结束（即当前层处理完毕），将 `level` 列表加入结果集 `res`。
   - 继续下一轮 `while` 循环，直到树中所有节点都被访问完毕。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n)$。其中 $n$ 是二叉树中的节点总数。每个节点都会且仅会入队和出队一次。
* **空间复杂度**：$O(n)$。主要消耗在队列上。在最坏情况下（完美二叉树），最后一层的节点数约为 $n/2$，队列需要存储这些节点。
:::

## 📝 题目 103：二叉树的锯齿形层序遍历

::: {.callout-caution}
### 题目要求
**描述**：给你二叉树的根节点 `root` ，返回其节点值的 **锯齿形层序遍历** 。（即先从左往右，下一层从右往左，以此类推，层与层之间交替进行）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`root = [3,9,20,null,null,15,7]`
    * **输出**：`[[3],[20,9],[15,7]]`

* **示例 2**：
    * **输入**：`root = [1]`
    * **输出**：`[[1]]`

* **示例 3**：
    * **输入**：`root = []`
    * **输出**：`[]`
:::

In [5]:
class Solution103:
    def zigzagLevelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if not root:  # 边界处理：如果根节点为空，直接返回空列表
            return []
        queue = deque([root])
        result = []
        direction = 1  # direction = 1 表示正向，-1 表示反向
        while queue:
            level = []
            for _ in range(len(queue)):
                node = queue.popleft()
                level.append(node.val)
                if node.left:  # 保持固定的入队顺序：左 -> 右
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            result.append(level[::direction])
            direction *= -1  # direction 为 1 时是 [::1] (原样)，为 -1 时是 [::-1] (反转)
        return result

In [7]:
#| code-fold: true
def test_103(func):
    test_cases = [
        {
            "input": [3, 9, 20, None, None, 15, 7],
            "expected": [[3], [20, 9], [15, 7]],
            "desc": "标准锯齿：左->右, 右->左, 左->右"
        },
        {
            "input": [1, 2, 3, 4, None, None, 5],
            "expected": [[1], [3, 2], [4, 5]],
            "desc": "非完全树锯齿"
        },
        {
            "input": [1],
            "expected": [[1]],
            "desc": "单节点"
        },
        {
            "input": [],
            "expected": [],
            "desc": "空树"
        },
        {
            "input": [1, 2, 3, 4, 5, 6, 7],
            "expected": [[1], [3, 2], [4, 5, 6, 7]],
            "desc": "满二叉树锯齿"
        },
        {
            "input": [1, 2, None, 3, None, 4],
            "expected": [[1], [2], [3], [4]],
            "desc": "全左斜树"
        },
        {
            "input": [1, None, 2, None, 3, None, 4],
            "expected": [[1], [2], [3], [4]],
            "desc": "全右斜树"
        },
        {
            "input": [3, 1, 4, None, 2],
            "expected": [[3], [4, 1], [2]],
            "desc": "小规模不平衡树"
        },
        {
            "input": [1, 2, 3, 4, None, None, 5, 6, None, None, 7],
            "expected": [[1], [3, 2], [4, 5], [7, 6]],
            "desc": "四层复杂树"
        },
        {
            "input": [0, 2, 4, 1, None, 3, -1, 5, 1, None, 6, None, 8],
            "expected": [[0], [4, 2], [1, 3, -1], [8, 6, 1, 5]],
            "desc": "含负数和大量节点的树"
        }
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 50)
    for i, tc in enumerate(test_cases):
        root = build_tree(tc["input"]) # 使用之前定义的辅助函数
        actual = func(root)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
        if not is_pass:
            print(f"   预期: {tc['expected']}\n   实际: {actual}")
    print("-" * 50)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_103(Solution103().zigzagLevelOrder)

ID   | 结果     | 描述
--------------------------------------------------
1    | ✅ PASS | 标准锯齿：左->右, 右->左, 左->右
2    | ✅ PASS | 非完全树锯齿
3    | ✅ PASS | 单节点
4    | ✅ PASS | 空树
5    | ✅ PASS | 满二叉树锯齿
6    | ✅ PASS | 全左斜树
7    | ✅ PASS | 全右斜树
8    | ✅ PASS | 小规模不平衡树
9    | ✅ PASS | 四层复杂树
10   | ✅ PASS | 含负数和大量节点的树
--------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **初始化与方向标记**：
   - 检查 `root` 是否为空。
   - 创建队列 `q` 并将 `root` 入队。
   - 引入一个方向标记 `direction`（你可以使用布尔值 `True/False` 或数字 `1/-1`）。初始状态为“正向”。

2. **层序遍历（标准 BFS）**：
   - 使用 `while q` 配合 `for _ in range(len(q))` 锁定当前层。
   - **物理顺序不变**：无论哪一层，入队顺序始终保持 **先左孩子、后右孩子**。这样可以确保树的物理结构在搜索过程中不被打乱。
   - **逻辑收集**：在 `for` 循环内部，将当前层的节点值顺序存入临时列表 `level`。

3. **锯齿形翻转**：
   - 处理完当前层的所有节点后，根据 `direction` 决定是否翻转 `level` 列表。
   - **技巧**：使用 Python 的切片语法 `level[::direction]`。
     - 若 `direction = 1`，结果为原顺序。
     - 若 `direction = -1`，结果为翻转顺序。
   - **翻转开关**：每一层结束后，改变 `direction` 的正负号（`direction *= -1`）。

:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n)$。每个节点访问一次，列表翻转的时间复杂度与层宽度成正比，总和仍为 $O(n)$。
* **空间复杂度**：$O(n)$。最坏情况下队列存储一层节点。
:::

## 📝 题目 199：二叉树的右视图

::: {.callout-caution}
### 题目要求
**描述**：给定一个二叉树的根节点 `root`，想象自己站在它的右侧，按照从顶部到底部的顺序，返回从右侧所能看到的节点值。

**输入输出示例**：

* **示例 1**：
    * **输入**：`root = [1,2,3,null,5,null,4]`
    * **输出**：`[1,3,4]`

* **示例 2**：
    * **输入**：`root = [1,null,3]`
    * **输出**：`[1,3]`

* **示例 3**：
    * **输入**：`root = []`
    * **输出**：`[]`
:::

In [8]:
class Solution199:
    def rightSideView(self, root: Optional[TreeNode]) -> List[int]:
        if not root:  # 边界处理：如果根节点为空，直接返回空列表
            return []
        queue = deque([root])
        result = []
        while queue:
            level = []
            for _ in range(len(queue)):
                node = queue.popleft()
                level.append(node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            result.append(level[-1])
        return result

In [9]:
#| code-fold: true
def test_199(func):
    test_cases = [
        {
            "input": [1, 2, 3, None, 5, None, 4],
            "expected": [1, 3, 4],
            "desc": "标准右视图"
        },
        {
            "input": [1, 2, 3, 4],
            "expected": [1, 3, 4],
            "desc": "左深右浅：4 虽然在左边，但在第三层是唯一的，所以可见"
        },
        {
            "input": [1, None, 3],
            "expected": [1, 3],
            "desc": "只有右支"
        },
        {
            "input": [],
            "expected": [],
            "desc": "空树"
        },
        {
            "input": [1, 2],
            "expected": [1, 2],
            "desc": "只有左子节点的树"
        },
        {
            "input": [1, 2, 3, 4, 5, 6, 7],
            "expected": [1, 3, 7],
            "desc": "满二叉树"
        },
        {
            "input": [1, 2, 3, None, None, None, 4, 5],
            "expected": [1, 3, 4, 5],
            "desc": "深层节点在左侧被挡住，但在更深层露出"
        },
        {
            "input": [3, 1, 4, 2],
            "expected": [3, 4, 2],
            "desc": "不规则树"
        },
        {
            "input": [1, 2, 3, 4, 5, None, None, 6],
            "expected": [1, 3, 5, 6],
            "desc": "多层遮挡测试"
        },
        {
            "input": [0, None, 0, None, 0],
            "expected": [0, 0, 0],
            "desc": "全 0 节点测试"
        }
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 50)
    for i, tc in enumerate(test_cases):
        root = build_tree(tc["input"]) # 使用之前定义的 build_tree 辅助函数
        actual = func(root)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
        if not is_pass:
            print(f"   预期: {tc['expected']}, 实际: {actual}")

    print("-" * 50)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_199(Solution199().rightSideView)

ID   | 结果     | 描述
--------------------------------------------------
1    | ✅ PASS | 标准右视图
2    | ✅ PASS | 左深右浅：4 虽然在左边，但在第三层是唯一的，所以可见
3    | ✅ PASS | 只有右支
4    | ✅ PASS | 空树
5    | ✅ PASS | 只有左子节点的树
6    | ✅ PASS | 满二叉树
7    | ✅ PASS | 深层节点在左侧被挡住，但在更深层露出
8    | ✅ PASS | 不规则树
9    | ✅ PASS | 多层遮挡测试
10   | ✅ PASS | 全 0 节点测试
--------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **层级遍历定位**：
   - 想象你站在树的右侧，你看到的其实是每一层中**位于最右侧**的那个节点。
   - 利用 BFS（广度优先搜索）进行层序遍历，可以确保我们逐层扫描树。

2. **识别“右侧”节点**：
   - 在处理每一层（即 `for _ in range(len(queue))` 循环）时，我们按从左到右的顺序弹出节点。
   - **关键点**：当循环运行到该层的**最后一个索引**时，当前弹出的节点就是该层“最右边”的节点。

3. **算法步骤**：
   - 初始化队列 `q` 并放入根节点。
   - 开启 `while` 循环，并在内部锁定当前层的大小 `size`。
   - 遍历当前层：
     - 弹出节点 `node`。
     - 如果这是该层的最后一个节点（即循环的最后一次迭代），将 `node.val` 加入结果集。
     - 正常将 `node.left` 和 `node.right` 加入队列。

:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n)$。每个节点访问一次。
* **空间复杂度**：$O(n)$。队列中最多存储树中宽度最大的一层节点。
:::